# **Libraries** **Preparation**

In [8]:
#Preparation for required libraries
!pip  install sklearn-crfsuite scikit-learn tabulate

import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.model_selection import train_test_split
from tabulate import tabulate

print("Libraries imported successfully!")


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Libraries imported successfully!


In [9]:
!pip  install sklearn-crfsuite


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


# **Data** **Loading**

In [18]:
import os

LOCAL_FILE_PATH = "mypos-ver.3.0.shuf.nopipe.txt"

# File check up!
if os.path.exists(LOCAL_FILE_PATH):
    with open(LOCAL_FILE_PATH, "r", encoding="utf-8") as f:
        # Dataset reading!
        dataset_content = f.readlines()

    print(f"File reading finished!This is total sentences here: {len(dataset_content)}")

    # first 5 sentences!
    print("\n--- First 5 Lines ---")
    for line in dataset_content[:5]:
        print(line.strip())
else:
    print(f"Error: '{LOCAL_FILE_PATH}' file don't fine!Please check again file path or name !")

File reading finished!This is total sentences here: 43196

--- First 5 Lines ---
၁၉၆၂/num ခုနှစ်/n ခန့်မှန်း/v သန်းခေါင်စာရင်း/n အရ/ppm လူဦးရေ/n ၁၁၅၉၃၁/num ယောက်/part ရှိ/v သည်/ppm ။/punc
လူ/n တိုင်း/part တွင်/ppm သင့်မြတ်/v လျော်ကန်/v စွာ/part ကန့်သတ်/v ထား/part သည့်/part အလုပ်/n လုပ်/v ချိန်/n အပြင်/conj ၊/punc လစာ/n နှင့်တကွ/conj အခါ/n ကာလ/n အားလျော်စွာ/ppm သတ်မှတ်/v ထား/part သည့်/part အလုပ်/n အားလပ်ရက်/n များ/part ပါဝင်/v သည့်/part အနားယူခွင့်/n နှင့်/conj အားလပ်ခွင့်/n ခံစားပိုင်ခွင့်/n ရှိ/v သည်/ppm ။/punc
ဤ/adj နည်း/n ကို/ppm စစ်ယူ/v သော/part နည်း/n ဟု/part ခေါ်/v သည်/ppm ။/punc
စာပြန်ပွဲ/n ဆို/v တာ/part က/ppm အာဂုံဆောင်/v အလွတ်ကျက်/v ထား/part တဲ့/part ပိဋကတ်သုံးပုံ/n စာပေ/n တွေ/part ကို/ppm စာစစ်/v သံဃာတော်ကြီး/n တွေ/part ရဲ့/ppm ရှေ့/n မှာ/ppm အလွတ်/adv ပြန်/v ပြီး/part ရွတ်ပြ/v ရ/part တာ/part ပေါ့/part ။/punc
ဒီ/pron မှာ/ppm ကျွန်တော့်/pron သက်သေခံကတ်/n ပါ/part ။/punc


# L&PLDF

In [11]:
def load_local_mypos_data(file_path):
    """
    Reads the local myPOS file line by line and parses tagged words into list of [(word, tag), ...] tuples.
    """
    sentences = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            tagged_words = line.split()
            sentence = []
            for item in tagged_words:
                # spreated word / tag!
                if '/' in item:
                    word, tag = item.rsplit('/', 1)
                    sentence.append((word, tag))

            if sentence:
                sentences.append(sentence)

    return sentences

# Dataset Loading from Local Disk
dataset = load_local_mypos_data(LOCAL_FILE_PATH)

print(f"Total sentences loaded: {len(dataset):,}")
print("Sample sentence 1:", dataset[0])

Total sentences loaded: 43,196
Sample sentence 1: [('၁၉၆၂', 'num'), ('ခုနှစ်', 'n'), ('ခန့်မှန်း', 'v'), ('သန်းခေါင်စာရင်း', 'n'), ('အရ', 'ppm'), ('လူဦးရေ', 'n'), ('၁၁၅၉၃၁', 'num'), ('ယောက်', 'part'), ('ရှိ', 'v'), ('သည်', 'ppm'), ('။', 'punc')]


# Feature Extraction Function

In [12]:
def word2features(sent, i):
    word = sent[i][0]

    features = {
        'bias': 1.0,
        'word': word,
        'word.len': len(word),
        'word.prefix-1': word[:1],
        'word.prefix-2': word[:2],
        'word.suffix-1': word[-1:],
        'word.suffix-2': word[-2:],
        'word.is_digit': word.isdigit(),
    }

    # Previous word features
    if i > 0:
        word1 = sent[i-1][0]
        features.update({
            '-1:word': word1,
            '-1:word.len': len(word1),
            '-1:word.suffix-1': word1[-1:],
        })
    else:
        features['BOS'] = True  # Begin of Sentence

    # Next word features
    if i < len(sent) - 1:
        word1 = sent[i+1][0]
        features.update({
            '+1:word': word1,
            '+1:word.len': len(word1),
            '+1:word.suffix-1': word1[-1:],
        })
    else:
        features['EOS'] = True  # End of Sentence

    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, label in sent]

# Train/Test Split& Feature Preparation

In [13]:
from sklearn.model_selection import train_test_split

# Train-Test Split (80% Train, 20% Test)
train_sents, test_sents = train_test_split(dataset, test_size=0.20, random_state=42)

X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]

X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]

print(f"Training set count: {len(X_train):,} sentences")
print(f"Testing set count:  {len(X_test):,} sentences")

Training set count: 34,556 sentences
Testing set count:  8,640 sentences


# CRF Model Traning

In [14]:
import sklearn_crfsuite

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,  # L1 regularization
    c2=0.1,  # L2 regularization
    max_iterations=100,
    all_possible_transitions=True
)

print("Training CRF POS Tagger Model...")
crf.fit(X_train, y_train)
print("Model training completed successfully!")

Training CRF POS Tagger Model...
Model training completed successfully!


# Model Evaluation

In [15]:
# Predict on testing set
y_pred = crf.predict(X_test)

labels = sorted(list(crf.classes_))

# Sequence Evaluation Metrics
flat_f1 = metrics.flat_f1_score(y_test, y_pred, average='weighted', labels=labels)
flat_accuracy = metrics.flat_accuracy_score(y_test, y_pred)

print(f"=== Model Evaluation Results ===")
print(f"Flat Accuracy: {flat_accuracy * 100:.2f}%")
print(f"Weighted F1-Score: {flat_f1 * 100:.2f}%\n")

# Detailed Tag-level Classification Report
print("=== Detailed Classification Report ===")
print(metrics.flat_classification_report(
    y_test, y_pred, labels=labels, digits=4
))

=== Model Evaluation Results ===
Flat Accuracy: 96.03%
Weighted F1-Score: 96.01%

=== Detailed Classification Report ===
              precision    recall  f1-score   support

         abb     0.9455    0.7536    0.8387        69
         adj     0.8474    0.8032    0.8247      3292
         adv     0.9091    0.8311    0.8684      2167
        conj     0.8940    0.9298    0.9115      3474
          fw     0.9752    0.9866    0.9809       598
         int     0.9398    0.9124    0.9259       137
           n     0.9587    0.9703    0.9645     24227
         num     0.9991    0.9957    0.9974      1174
        part     0.9659    0.9665    0.9662     26702
         ppm     0.9796    0.9837    0.9817     17029
        pron     0.9619    0.9610    0.9614      4021
        punc     0.9988    0.9991    0.9989     10782
          sb     1.0000    0.8103    0.8952        58
          tn     0.9784    0.9650    0.9716      1172
           v     0.9452    0.9387    0.9419     16611

    accuracy 

# Testing Sample Sentence

In [16]:
def pos_tag_sentence(sentence_words, model):
    dummy_sent = [(w, 'UNK') for w in sentence_words]
    features = sent2features(dummy_sent)
    predicted_tags = model.predict_single(features)
    return list(zip(sentence_words, predicted_tags))

# Testing custom sentence
#sample_words = ['ကျွန်တော်', 'မမ', 'ရဲ့','အလှ','ကို', 'မြတ်နိုး', 'သည်', '။']
#sample_words = ['ကျောင်းသား', 'များ', 'သည်', 'စာကြည့်တိုက်', 'တွင်', 'စာဖတ်', 'နေ', 'ကြ', 'သည်', '။']
#sample_words = ['ယနေ့', 'မနက်ပိုင်း', 'တွင်', 'မိုး', 'သည်းထန်စွာ', 'ရွာသွန်း', 'ခဲ့', 'သည်', '။']
#sample_words = ['ဆရာမ', 'က', 'စာအုပ်', 'ကို', 'ဖတ်ပြ', 'ပြီး', 'မေးခွန်း', 'များ', 'မေး', 'ခဲ့', 'သည်', '။']
#sample_words = ['ဆရာမ', 'က', 'ပေတံ', 'ဘုရင်မ', 'ဖြစ်ပြီး', 'စာပွဲ', 'ကို', 'ဒုန်း', 'ခနဲ', 'ရိုက်', 'လိုက်', 'သည်', '။']
#sample_words = ['စာမရသော', 'ကျောင်းသား', 'ကို', 'ဆရာမ', 'က', 'ဒေါသ', 'ထွက်စွာ', 'စိုက်ကြည့်', 'နေ', 'သည်', '။']
#sample_words = ['စကားပြော', 'သော', 'မောင်မောင်', '၏', 'နားရွက်', 'ကို', 'ဆရာမ', 'က', 'ဆွဲ', 'လိုက်', 'သည်', '။']
#sample_words = ['အိမ်စာ', 'မပါသော', 'သူများ', 'ကို', 'ဆရာမ', 'က', 'မတ်တပ်', 'ရပ်', 'ခိုင်း', 'ထား', 'သည်', '။']
sample_words = ['စည်းကမ်း', 'မရှိသော', 'ကျောင်းသား', 'ကို','ဓာတုဗေဒ','ဆရာမ', 'က', 'ဓာတ်ခွဲခန်း', 'ထဲ', 'မှ', 'နှင်ထုတ်', 'လိုက်', 'သည်', '။']


In [17]:
from tabulate import tabulate

# POS Tag mean and dictionary!
tag_descriptions = {
    'n'   :   'Noun (နာမ်)',
    'v'   :   'Verb (ကိရိယာ)',
    'ppm' :   'Post-position Marker (ဝိဘတ်)',
    'part':   'Particle (ပစ္စည်း)',
    'pron':   'Pronoun (နာမ်စား)',
    'adj' :   'Adjective (နာမဝိသေသန)',
    'adv' :   'Adverb (ကြိယာဝိသေသန)',
    'punc':   'Punctuation (ပုဒ်ဖြတ်ပုဒ်ရပ်)',
    'num' :   'Number (ဂဏန်း)',
    'conj':   'Conjunction (စကားဆက်)'
}

results = pos_tag_sentence(sample_words, crf)

# Data Preparation for tabel view!
table_data = []
for word, tag in results:
    meaning = tag_descriptions.get(tag, tag)
    table_data.append([word, tag, meaning])

# print final result with table version!
headers = ["Word (စကားလုံး)", "POS Tag", "Meaning (အဓိပ္ပာယ်)"]
print("\nSample Prediction Output:")
print(tabulate(table_data, headers=headers, tablefmt="grid"))


Sample Prediction Output:
+----------------+-----------+----------------------------+
| Word (စကားလုံး)   | POS Tag   | Meaning (အဓိပ္ပာယ်)            |
+================+===========+============================+
| စည်းကမ်း         | n         | Noun (နာမ်)                 |
+----------------+-----------+----------------------------+
| မရှိသော           | n         | Noun (နာမ်)                 |
+----------------+-----------+----------------------------+
| ကျောင်းသား         | n         | Noun (နာမ်)                 |
+----------------+-----------+----------------------------+
| ကို              | ppm       | Post-position Marker (ဝိဘတ်) |
+----------------+-----------+----------------------------+
| ဓာတုဗေဒ         | n         | Noun (နာမ်)                 |
+----------------+-----------+----------------------------+
| ဆရာမ           | n         | Noun (နာမ်)                 |
+----------------+-----------+----------------------------+
| က              | ppm       | Post-position Ma